<div style='text-align: center; padding: 30px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); border-radius: 15px; margin: 10px 0; box-shadow: 0 10px 30px rgba(0,0,0,0.2);'>
  <h1 style='color: white; margin: 0 0 8px 0; font-size: 2.5em;'>🎤 AuK v1.0 - Unified Audio Generation and Editing</h1>
  <h3 style='color: #f0f0f0; margin: 0 0 5px 0; font-weight: 400;'>Kaggle T4 x2 GPU Edition - Created by <strong>AIQUEST Academy</strong></h3>
  <p style='color: #ddd; margin: 0; text-align: center;'>1.5B Foundational Base Audio Model for High-Quality Audio Generation & Editing</p>
</div>

<div align="center">
  <img src="https://img.shields.io/badge/AIQUESTAcademy-blueviolet?style=for-the-badge&logo=youtube&logoColor=white" />
  <img src="https://img.shields.io/badge/Kaggle-T4%20GPU%20x2-20BEFF?style=for-the-badge&logo=kaggle&logoColor=white" />
  <br>
  <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1">
    <img src="https://img.shields.io/badge/Subscribe%20on%20YouTube-FF0000?style=for-the-badge&logo=youtube&logoColor=white" />
  </a>
  &nbsp;
  <a href="https://x.com/aiquestacademy">
    <img src="https://img.shields.io/badge/Follow%20on%20X-000000?style=for-the-badge&logo=x&logoColor=white" />
  </a>
</div>

### Setup Instructions
1. **Settings -> Accelerator -> GPU T4 x2** (MUST select dual T4 GPUs for zero-offload instant inference)
2. Run all cells **top to bottom** (Cell 1 -> Cell 2 -> Cell 3 -> Cell 4)
3. Open the public Gradio link generated in Cell 4

---

## ⚙️ Cell 1 - Environment and Dual-GPU Memory Configuration

In [ ]:
# Cell 1: Check GPU and Optimize Environment Memory
from __future__ import annotations
import os
import gc
import sys
import torch
import psutil

print("=== Kaggle T4 Environment Setup ===")
print(f"Python Version: {sys.version}")
print(f"PyTorch Version: {torch.__version__}")
print(f"RAM: {psutil.virtual_memory().total / 1024**3:.1f} GB total, {psutil.virtual_memory().available / 1024**3:.1f} GB available")

# Optimize virtual memory allocations and drop page caches
os.system("echo 3 | sudo tee /proc/sys/vm/drop_caches > /dev/null 2>&1")
os.system("echo 1 | sudo tee /proc/sys/vm/overcommit_memory > /dev/null 2>&1")
gc.collect()

# Prevent memory fragmentation and aggressive VRAM allocation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,garbage_collection_threshold:0.6"
os.environ["MALLOC_TRIM_THRESHOLD_"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

if torch.cuda.is_available():
    device_count = torch.cuda.device_count()
    print(f"Number of GPUs Available: {device_count}")
    for i in range(device_count):
        device_name = torch.cuda.get_device_name(i)
        total_memory = torch.cuda.get_device_properties(i).total_memory / 1e9
        print(f"  GPU {i}: {device_name} ({total_memory:.2f} GB VRAM)")
    if device_count < 2:
        print("WARNING: Only 1 GPU detected. Go to Settings -> Accelerator -> GPU T4 x2 to enable dual GPUs.")
    else:
        print("SUCCESS: Dual GPU T4 x2 detected! High-throughput cross-GPU pipeline will be enabled.")
else:
    print("WARNING: No GPU detected. Go to Settings -> Accelerator -> GPU T4 x2")

# Force efficient attention options for Turing GPUs (T4)
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)

print("Environment setup and memory footprint optimizations applied!")

## 📦 Cell 2 - Install System & Python Dependencies

In [ ]:
# Cell 2: Install required packages and clone repository
import os
import sys

print("Installing system audio packages (ffmpeg, libsndfile1-dev)...")
os.system("apt-get update -qq && apt-get install -y -qq ffmpeg libsndfile1-dev > /dev/null 2>&1")

print("Cloning official AuK repository...")
if not os.path.exists("AuK"):
    os.system("git clone --depth 1 https://github.com/Tencent-Hunyuan/AuK")

auk_src = os.path.abspath("AuK/src")
if auk_src not in sys.path:
    sys.path.insert(0, auk_src)

print("Installing Python dependencies for AuK, Transformers, and Gradio...")
os.system("pip install -q 'transformers>=4.52.0,<5' qwen-omni-utils torchdiffeq x_transformers omegaconf accelerate safetensors gradio pyloudnorm soundfile hf-transfer pydantic")

print("Dependencies installed successfully!")

## 📥 Cell 3 - Fast Parallel Checkpoint Acquisition

In [ ]:
# Cell 3: Download model weights with hf_transfer
import os
import sys

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
from huggingface_hub import snapshot_download

# Which DiT to fetch. Tencent's model table: AuK = "Base model for high-quality generation",
# AuK-Flash = "Distilled model for fast 4-step inference". Base runs 32 steps with CFG=2.0
# (64 DiT evaluations per clip) vs Flash's 4 steps with CFG off, so base is far slower on a T4
# but is the higher-fidelity model. Cell 4 auto-detects whichever variant is present.
#   "base"  -> quality  (recommended)
#   "flash" -> speed
#   "both"  -> download both, Cell 4 prefers base
AUK_VARIANT = "base"

AUK_REPOS = {
    "base": ("tencent/AuK", "ckpts/AuK", "auk_base.safetensors", "base DiT, 32-step + CFG"),
    "flash": ("tencent/AuK-Flash", "ckpts/AuK-Flash", "auk_flash.safetensors", "4-step distilled DiT"),
}
wanted = ["base", "flash"] if AUK_VARIANT == "both" else [AUK_VARIANT]
assert all(w in AUK_REPOS for w in wanted), f"AUK_VARIANT must be base, flash or both (got {AUK_VARIANT!r})"

print("=== Downloading Model Checkpoints ===")
os.makedirs("ckpts/Qwen2.5-Omni-3B", exist_ok=True)

verify_paths = []
for i, key in enumerate(wanted, start=1):
    repo_id, local_dir, weight_file, note = AUK_REPOS[key]
    os.makedirs(local_dir, exist_ok=True)
    print(f"{i}. Downloading {repo_id} ({note} + BigVGAN VAE)...")
    snapshot_download(
        repo_id=repo_id,
        local_dir=local_dir,
        allow_patterns=["*.safetensors", "*.yaml", "*.json", "*.txt"],
    )
    verify_paths += [f"{local_dir}/{weight_file}", f"{local_dir}/vae.safetensors", f"{local_dir}/config.yaml"]

print(f"{len(wanted) + 1}. Downloading Qwen2.5-Omni-3B MLLM Encoder...")
snapshot_download(
    repo_id="Qwen/Qwen2.5-Omni-3B",
    local_dir="ckpts/Qwen2.5-Omni-3B",
    allow_patterns=["*.safetensors", "*.json", "*.txt", "tokenizer*"],
)

print("\nVerifying checkpoint directory structure:")
for path in verify_paths + ["ckpts/Qwen2.5-Omni-3B/config.json"]:
    if os.path.isfile(path):
        size_mb = os.path.getsize(path) / 1024**2
        print(f"  [OK] {path} ({size_mb:.1f} MB)")
    else:
        print(f"  [MISSING] {path}")

print("All checkpoints verified successfully!")

## 🚀 Cell 4 - AuK Dual-GPU Engine & Gradio Web Interface

In [ ]:
# Cell 4: Initialize AuK Dual-GPU Engine and Launch Gradio UI
import os
import gc

import re
import sys
import time
import math
import random
import torch
import torchaudio
import gradio as gr
from omegaconf import OmegaConf
from torch.nn.utils.rnn import pad_sequence
from torchdiffeq import odeint

# Ensure real-time unbuffered stdout so logs stream immediately in Kaggle cell output
try:
    sys.stdout.reconfigure(line_buffering=True)
    sys.stderr.reconfigure(line_buffering=True)
except Exception:
    pass

auk_src = os.path.abspath("AuK/src")
if auk_src not in sys.path:
    sys.path.insert(0, auk_src)

from auk.infer.infer_auk import AukInfer
import torch.nn.functional as F

print("=== Initializing AuK Engine ===", flush=True)

# Detect available devices
num_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
if num_gpus >= 2:
    device_main = "cuda:1"
    device_te = "cuda:0"
    print(f"Dual-GPU setup active: Qwen2.5-Omni on {device_te}, AuK-Flash DiT + VAE on {device_main}", flush=True)
elif num_gpus == 1:
    device_main = "cuda:0"
    device_te = "cuda:0"
    print(f"Single-GPU setup active on {device_main}", flush=True)
else:
    device_main = "cpu"
    device_te = "cpu"
    print("Warning: Running on CPU", flush=True)

# Resolve which DiT to run. "auto" picks whatever Cell 3 downloaded, preferring the base model.
# Set to "base" or "flash" to force one when both are present.
AUK_VARIANT = "auto"

AUK_VARIANTS = {
    "base": {
        "label": "AuK (Base)",
        "config": "ckpts/AuK/config.yaml",
        "ckpt": "ckpts/AuK/auk_base.safetensors",
        # infer_gradio.py SAMPLING_PRESETS[BASE_LABEL]. 32 is the official default; the UI
        # slider goes down to 4 if you ever want to trade quality for speed.
        "nfe": 32,
        "cfg": 2.0,
    },
    "flash": {
        "label": "AuK-Flash (Distilled)",
        "config": "ckpts/AuK-Flash/config.yaml",
        "ckpt": "ckpts/AuK-Flash/auk_flash.safetensors",
        # infer_gradio.py SAMPLING_PRESETS[FLASH_LABEL]; AukInfer re-pins these internally anyway
        "nfe": 4,
        "cfg": 0.0,
    },
}

if AUK_VARIANT == "auto":
    found = [k for k in ("base", "flash") if os.path.isfile(AUK_VARIANTS[k]["ckpt"])]
    if not found:
        raise FileNotFoundError(
            "No AuK checkpoint found under ckpts/. Run Cell 3 first "
            f"(looked for {AUK_VARIANTS['base']['ckpt']} and {AUK_VARIANTS['flash']['ckpt']})."
        )
    AUK_VARIANT = found[0]

variant_cfg = AUK_VARIANTS[AUK_VARIANT]
config_path = variant_cfg["config"]
ckpt_path = variant_cfg["ckpt"]
qwen_path = "ckpts/Qwen2.5-Omni-3B"
print(f"Model variant: {variant_cfg['label']}  ({ckpt_path})", flush=True)

# Initialize official AukInfer engine with dtype="fp32"
# Crucial for NVIDIA T4 (Turing): Avoids broken bfloat16 emulation and prevents fp16 overflow
engine = AukInfer(
    config_path=config_path,
    ckpt_path=ckpt_path,
    device=device_main,
    dtype="fp32",
    qwen_path=qwen_path,
    cpu_offload=False,
)

# Ensure engine.model.device always maps to device_main (cuda:1) to prevent device mismatch errors
engine.model.layer_weights.data = engine.model.layer_weights.data.to(device=device_main, dtype=torch.float32)
engine.model.layer_scale.data = engine.model.layer_scale.data.to(device=device_main, dtype=torch.float32)
type(engine.model).device = property(lambda self: torch.device(device_main))

# Distribute Qwen LLM to GPU 0 in native BFloat16 (avoids FP16 attention logit overflow >65504)
# Layer fusion and all subsequent DiT computation are performed in pure FP32
if num_gpus >= 2:
    print(f"Distributing Qwen text encoder to {device_te} in native BFloat16 for dual-GPU load balancing...", flush=True)
    engine.model.text_encoder = engine.model.text_encoder.to(device=device_te, dtype=torch.bfloat16)

    def dual_gpu_encode_text(cond_inputs, target_device):
        if hasattr(cond_inputs, "to"):
            cond_inputs = cond_inputs.to(device_te)
        else:
            cond_inputs = {k: (v.to(device_te) if torch.is_tensor(v) else v) for k, v in cond_inputs.items()}
        for k, v in cond_inputs.items():
            if torch.is_tensor(v) and v.is_floating_point():
                cond_inputs[k] = v.to(torch.bfloat16)
        attention_mask = cond_inputs["attention_mask"]
        with torch.no_grad():
            outputs = engine.model.text_encoder(**cond_inputs, output_hidden_states=True)
        all_hidden_states = outputs.hidden_states
        _, _, d_llm = all_hidden_states[0].shape
        stacked = torch.stack([F.layer_norm(h.float(), [d_llm]) for h in all_hidden_states[1:]], dim=0)
        weights = F.softmax(engine.model.layer_weights.to(device_te).float(), dim=0)
        hidden = (stacked * weights[:, None, None, None]).sum(dim=0) * engine.model.layer_scale.to(device_te).float()
        return hidden.to(device=target_device, dtype=torch.float32), attention_mask.bool().to(target_device)

    engine.model.encode_text = dual_gpu_encode_text
else:
    engine.model.text_encoder = engine.model.text_encoder.to(device=device_main, dtype=torch.bfloat16)

    def single_gpu_encode_text(cond_inputs, target_device):
        if hasattr(cond_inputs, "to"):
            cond_inputs = cond_inputs.to(device_main)
        else:
            cond_inputs = {k: (v.to(device_main) if torch.is_tensor(v) else v) for k, v in cond_inputs.items()}
        for k, v in cond_inputs.items():
            if torch.is_tensor(v) and v.is_floating_point():
                cond_inputs[k] = v.to(torch.bfloat16)
        attention_mask = cond_inputs["attention_mask"]
        with torch.no_grad():
            outputs = engine.model.text_encoder(**cond_inputs, output_hidden_states=True)
        all_hidden_states = outputs.hidden_states
        _, _, d_llm = all_hidden_states[0].shape
        stacked = torch.stack([F.layer_norm(h.float(), [d_llm]) for h in all_hidden_states[1:]], dim=0)
        weights = F.softmax(engine.model.layer_weights.float(), dim=0)
        hidden = (stacked * weights[:, None, None, None]).sum(dim=0) * engine.model.layer_scale.float()
        return hidden.to(device=target_device, dtype=torch.float32), attention_mask.bool().to(target_device)

    engine.model.encode_text = single_gpu_encode_text

# Explicit safe_sample patch ensuring all DiT integration tensors are strictly co-located on device_main (cuda:1)
@torch.no_grad()
def safe_sample(
    self,
    cond: torch.Tensor,
    text,
    duration: int | torch.Tensor,
    *,
    lens: torch.Tensor | None = None,
    steps=32,
    cfg_strength=1.0,
    sway_sampling_coef=None,
    t_grid: list[float] | None = None,
    seed: int | None = None,
    max_duration=65536,
    vocoder=None,
    use_epss=True,
    no_ref_audio=False,
):
    self.eval()
    dev = torch.device(device_main)
    dtype_work = torch.float32

    cond = cond.to(device=dev, dtype=dtype_work)
    batch, cond_seq_len = cond.shape[:2]

    ref_latent = cond
    if lens is None:
        ref_lens = torch.full((batch,), cond_seq_len, device=dev, dtype=torch.long)
    else:
        ref_lens = lens.to(device=dev, dtype=torch.long)

    ref_mask = (torch.arange(cond_seq_len, device=dev) < ref_lens[:, None])

    if no_ref_audio:
        ref_latent = torch.zeros_like(ref_latent)

    # Encode text on GPU 0 and move embeddings to GPU 1
    text_embeds, context_mask = self.encode_text(text, dev)
    text_embeds = text_embeds.to(device=dev, dtype=dtype_work)
    context_mask = context_mask.to(device=dev)

    if isinstance(duration, int):
        duration = torch.full((batch,), duration, device=dev, dtype=torch.long)
    else:
        duration = duration.to(device=dev, dtype=torch.long)

    duration = duration.clamp(max=max_duration)
    target_duration = (duration - ref_lens).clamp(min=1)
    duration = ref_lens + target_duration

    if batch > 1:
        max_td = target_duration.amax().item()
        target_mask = (torch.arange(max_td, device=dev) < target_duration[:, None])
    else:
        target_mask = None

    def fn(t, x):
        x = x.to(dev)
        t = t.to(dev)
        if cfg_strength < 1e-5:
            return self.transformer(
                x=x,
                text=text_embeds,
                time=t,
                mask=target_mask,
                c_mask=context_mask,
                ref=ref_latent,
                ref_mask=ref_mask,
                drop_audio_cond=False,
                drop_text=False,
                cache=True,
            )
        pred_cfg = self.transformer(
            x=x,
            text=text_embeds,
            time=t,
            mask=target_mask,
            c_mask=context_mask,
            ref=ref_latent,
            ref_mask=ref_mask,
            cfg_infer=True,
            cache=True,
        )
        v_cond, v_uncond = torch.chunk(pred_cfg, 2, dim=0)
        return v_cond + (v_cond - v_uncond) * cfg_strength

    cuda_gen = None
    if seed is not None and int(seed) >= 0:
        eff_s = int(seed)
        torch.manual_seed(eff_s)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(eff_s)
        if dev.type == "cuda":
            cuda_gen = torch.Generator(device=dev).manual_seed(eff_s)
        else:
            cuda_gen = torch.Generator(device="cpu").manual_seed(eff_s)

    y0 = []
    for dur in target_duration:
        if cuda_gen is not None:
            y0.append(torch.randn(dur.item(), self.num_channels, device=dev, dtype=dtype_work, generator=cuda_gen))
        else:
            y0.append(torch.randn(dur.item(), self.num_channels, device=dev, dtype=dtype_work))
    y0 = pad_sequence(y0, padding_value=0, batch_first=True)

    t = torch.linspace(0, 1, steps + 1, device=dev, dtype=torch.float32)
    if t_grid is not None:
        t = torch.tensor(t_grid, device=dev, dtype=torch.float32)
    elif sway_sampling_coef is not None:
        t = t + sway_sampling_coef * (torch.cos(torch.pi / 2 * t) - 1 + t)

    trajectory = odeint(fn, y0, t, **self.odeint_kwargs)
    self.transformer.clear_cache()
    sampled = trajectory[-1]

    total_max_dur = duration.amax().item()
    out = torch.zeros(batch, total_max_dur, self.num_channels, device=dev, dtype=sampled.dtype)
    for i in range(batch):
        rl = ref_lens[i].item()
        out[i, :rl] = ref_latent[i, :rl]
        td = target_duration[i].item()
        out[i, rl : rl + td] = sampled[i, :td]

    if vocoder is not None:
        out = out.permute(0, 2, 1)
        out = vocoder(out)

    return out, trajectory

engine.model.sample = safe_sample.__get__(engine.model, type(engine.model))

# Patch _load_audio to trim input audio to exact multiple of 480 samples (eliminates 505 vs 504 mismatch)
orig_load_audio = engine._load_audio

def safe_load_audio(source):
    audio, rms = orig_load_audio(source)
    num_frames = audio.shape[-1] // engine.downsample_rate
    if num_frames > 0:
        audio = audio[:, : num_frames * engine.downsample_rate]
    return audio, rms

engine._load_audio = safe_load_audio

# Patch _run to ensure clean Float32 execution without broken bfloat16 autocasting on T4
@torch.inference_mode()
def safe_run(
    ref_audio: torch.Tensor,
    ref_rms: float | None,
    messages: list,
    gen_latent_len: int,
    *,
    nfe: int,
    cfg_strength: float,
    sway_sampling_coef: float,
    t_grid: list[float] | None,
    seed: int | None,
) -> torch.Tensor:
    dev = torch.device(device_main)
    if ref_rms is None:
        ref_latent_lens_t = torch.zeros(1, dtype=torch.long, device=dev)
        total_latent_lens_t = torch.tensor([gen_latent_len], dtype=torch.long, device=dev)
        ref_latents = torch.zeros(1, 0, engine.latent_dim, device=dev, dtype=torch.float32)
    else:
        ref_audio = ref_audio.to(dev).unsqueeze(0)  # [1, 1, T]
        ref_latent_len = ref_audio.shape[-1] // engine.downsample_rate
        total_latent_len = ref_latent_len + gen_latent_len

        ref_latent_lens_t = torch.tensor([ref_latent_len], dtype=torch.long, device=dev)
        total_latent_lens_t = torch.tensor([total_latent_len], dtype=torch.long, device=dev)
        audio_lens_t = ref_latent_lens_t * engine.downsample_rate

        # Online VAE encode + normalize
        ref_latents, enc_latent_lens = engine.vae_model.encoding_and_normalization(
            ref_audio,
            sample_lengths=audio_lens_t,
        )
        actual_ref_len = enc_latent_lens[0].item()
        ref_latent_lens_t = torch.tensor([actual_ref_len], dtype=torch.long, device=dev)
        total_latent_lens_t = torch.tensor([actual_ref_len + gen_latent_len], dtype=torch.long, device=dev)
        if ref_latents.shape[1] > actual_ref_len:
            ref_latents = ref_latents[:, :actual_ref_len, :]

    # CFM sample in latent space in clean Float32 (No T4 bf16 bugs, No fp16 overflow)
    cond_inputs = engine.model.build_cond_inputs([messages], engine.model.text_processor)
    generated, _ = engine.model.sample(
        cond=ref_latents,
        text=cond_inputs,
        duration=total_latent_lens_t,
        lens=ref_latent_lens_t,
        steps=nfe,
        cfg_strength=cfg_strength,
        sway_sampling_coef=sway_sampling_coef,
        t_grid=t_grid,
        no_ref_audio=False,
        seed=seed,
    )

    gen = generated[0]
    rl = ref_latent_lens_t[0].item()
    tl = total_latent_lens_t[0].item()
    gen_latent = gen[rl:tl, :].unsqueeze(0)
    if gen_latent.shape[1] == 0:
        raise RuntimeError("Empty generated latent (target duration collapsed to 0).")
    if torch.isnan(gen_latent).any() or torch.isinf(gen_latent).any():
        raise RuntimeError("Generated latent contains NaN/Inf.")

    gen_latent = engine.vae_model.denormalize(gen_latent)
    gen_latent = gen_latent.permute(0, 2, 1)

    gen_audio = engine.vae_model.inference_from_latents(gen_latent).cpu()
    if gen_audio.ndim == 3:
        gen_audio = gen_audio.squeeze(0)
    if torch.isnan(gen_audio).any() or torch.isinf(gen_audio).any():
        raise RuntimeError("Generated audio contains NaN/Inf.")

    return gen_audio.to(torch.float32)

engine._run = safe_run

# AukInfer overrides nfe/cfg internally when config.model.name == "AuK-Flash". Print what is
# actually in force so a mislabelled config cannot silently run base settings on the distilled
# checkpoint (which, per infer_auk.py, "blows up the amplitude (clips hard)").
if engine.is_flash:
    AUK_NFE, AUK_CFG = 4, 0.0
    print("Sampling recipe: 4 steps, CFG off (pinned by AukInfer for the distilled checkpoint)", flush=True)
else:
    AUK_NFE, AUK_CFG = variant_cfg["nfe"], variant_cfg["cfg"]
    print(f"Sampling recipe: {AUK_NFE} steps, CFG {AUK_CFG} (engine.is_flash=False)", flush=True)
if AUK_VARIANT == "flash" and not engine.is_flash:
    print("WARNING: flash checkpoint loaded but engine.is_flash is False - check config.model.name.", flush=True)

print("AuK Engine fully loaded with robust FP32 inference pipeline!", flush=True)

# Official AuK Instruction Templates & Duration Calibration
#
# Source of truth: Tencent-Hunyuan/AuK @ src/auk/infer/pe.config.yaml
#   - tasks.instruct_tts.templates  -> the only Instruct-TTS strings the model is trained on
#   - tasks.zero_shot_tts.templates -> the zero-shot (voice cloning) string
#   - runtime.duration              -> the F5 duration baseline used by the Prompt Enhancer
#
# Anti-leakage contract: AuK is a non-autoregressive flow-matching DiT with no EOS token, so
# gen_seconds alone decides how many latent frames get decoded. Every token in the prompt is a
# candidate for vocalization, and any duration slack past the end of the target text WILL be
# filled by decoding the style description aloud. Two invariants prevent that:
#   1. The style description must sit in the template's description slot (never in the content
#      slot, and never in a free-form `Say "<text>" in <style>` sentence, which the model treats
#      as one continuous utterance).
#   2. gen_seconds must equal the *style-adjusted natural length* of the target text alone.
AUK_INSTRUCT_TTS_TEMPLATE = {
    "en": 'Based on the following description: "{style}", generate speech content "{text}".',
    "zh": '请基于下面的描述: "{style}",生成语音内容"{text}".',
}
AUK_ZERO_SHOT_TTS_TEMPLATE = 'Say the following with the same voice: "{text}"'

# runtime.duration.seconds_per_utf8_byte / runtime.duration.f5 / runtime.duration.output_frames_per_second
AUK_SEC_PER_UTF8_BYTE = {"en": 0.0656, "zh": 0.0803}
AUK_F5_SAMPLE_RATE = 24000
AUK_F5_HOP_LENGTH = 256
AUK_F5_SHORT_TEXT_BYTE_THRESHOLD = 10
AUK_F5_SHORT_TEXT_SPEED = 0.3
AUK_LATENT_FRAMES_PER_SECOND = 50

# The PE rejects any LLM duration outside this ratio band relative to the F5 baseline.
AUK_RATIO_MIN, AUK_RATIO_MAX = 0.45, 2.20

# The F5 baseline is calibrated for reading speed, and AuK's own demo durations for neutral
# delivery land consistently below it (measured ratios 0.79 / 0.87 / 0.92 across the official
# examples). This constant re-centres the baseline on AuK's actual neutral pace; the cue table
# below then moves it. Lower it if you still hear description leakage in the tail.
AUK_NEUTRAL_PACE = 0.88

# Timing cues only. The official duration prompt is explicit: "Do not adjust for emotion, pitch,
# volume, age, or accent unless the description explicitly implies timing behavior." So "mournful",
# "solemn", "grief" and "sad" contribute nothing on their own -- but "slow", "pauses", "breaking"
# and "choked" do, because they change how long the words take to say.
AUK_TIMING_CUES = (
    (0.30, r"slow|slowly|slower|unhurried|leisurely|measured|deliberate|缓慢|缓缓|放慢|舒缓|从容|语速慢"),
    (0.28, r"pause|pauses|pausing|halting|broken|breaking|停顿|断断续续"),
    (0.12, r"hesitat|stammer|stutter|falter|unsteady breath|犹豫|结巴|气息不稳"),
    (0.14, r"chok(?:e|ed|ing)|sob|sobbing|weeping|哽咽|抽泣|哭腔"),
    (0.12, r"elongat|drawn[- ]out|lingering|拖长|拉长"),
    (0.40, r"\bsing|singing|sung|lyric|吟唱|歌唱|演唱"),
    (-0.25, r"fast|quick|rapid|rushed|hurried|breathless|brisk|快速|急促|语速快|加快"),
    (-0.15, r"clipped|staccato|terse|curt|短促|干脆利落"),
)

_AUK_CJK_RE = re.compile(r"[㐀-䶿一-鿿豈-﫿]")
_AUK_EN_WORD_RE = re.compile(r"[A-Za-z]+")
# Slot delimiters. Straight/curly double quotes only -- a bare apostrophe is excluded so that
# contractions like "he'd" cannot terminate a slot early.
_AUK_DQ = "\"“”"
_AUK_Q = _AUK_DQ + "'‘’"


def _auk_text_language(text: str, fallback: str = "en") -> str:
    """Port of pe._resolve_text_language: pick zh/en by weighted script content."""
    value = str(text or "")
    num_zh = len(_AUK_CJK_RE.findall(value))
    num_en = len(_AUK_EN_WORD_RE.findall(value))
    if num_zh and not num_en:
        return "zh"
    if num_en and not num_zh:
        return "en"
    if num_zh and num_en:
        return "zh" if num_zh * 0.21 >= num_en * 0.30 else "en"
    return fallback


def _auk_utf8_weight(text: str, fallback: str) -> float:
    """Port of pe._tts_utf8_weight: per-character seconds, with script carried across punctuation."""
    value = str(text or "")
    if not value.strip():
        return 0.0
    script = [
        "zh" if _AUK_CJK_RE.fullmatch(ch) else ("en" if ch.isascii() and ch.isalpha() else None)
        for ch in value
    ]
    following = [None] * len(value)
    nxt = None
    for i in range(len(value) - 1, -1, -1):
        if script[i]:
            nxt = script[i]
        following[i] = nxt
    weight = 0.0
    prev = None
    for i, ch in enumerate(value):
        lang = script[i]
        if lang is None:
            lang = prev or following[i] or fallback
        else:
            prev = lang
        weight += len(ch.encode("utf-8")) * AUK_SEC_PER_UTF8_BYTE[lang]
    return weight


def _auk_f5_baseline(text: str, language: str) -> float:
    """Port of pe._estimate_f5_instruct_duration: neutral reading length of the target text."""
    weight = _auk_utf8_weight(text, language)
    speed = AUK_F5_SHORT_TEXT_SPEED if len(str(text or "").encode("utf-8")) < AUK_F5_SHORT_TEXT_BYTE_THRESHOLD else 1.0
    frames = int(weight * AUK_F5_SAMPLE_RATE / AUK_F5_HOP_LENGTH / speed)
    return frames * AUK_F5_HOP_LENGTH / AUK_F5_SAMPLE_RATE


def _auk_pace_ratio(style: str) -> float:
    """Multiplier on the F5 baseline, driven only by cues that change speaking time."""
    s = str(style or "").lower()
    ratio = 1.0
    for delta, pattern in AUK_TIMING_CUES:
        if re.search(pattern, s):
            ratio += delta
    return min(max(ratio * AUK_NEUTRAL_PACE, AUK_RATIO_MIN), AUK_RATIO_MAX)


def _auk_quantize_seconds(seconds: float) -> float:
    """Port of pe._quantize_model_duration.

    AukInfer.generate() converts seconds to frames with math.ceil(seconds * 50). Feeding it a
    float that is one ULP above an exact frame boundary buys an extra latent frame, so nudge the
    value down until ceil() agrees with round().
    """
    target_len = max(1, round(float(seconds) * AUK_LATENT_FRAMES_PER_SECOND))
    nominal = target_len / AUK_LATENT_FRAMES_PER_SECOND
    if math.ceil(nominal * AUK_LATENT_FRAMES_PER_SECOND) != target_len:
        nominal = math.nextafter(nominal, 0.0)
    return nominal


def _auk_strip_slot(value: str) -> str:
    """Remove slot-delimiting quotes from a slot value, preserving apostrophes.

    A double quote inside the description slot closes it early, which promotes the remainder of
    the description into the content slot -- i.e. it gets spoken. Apostrophes are safe (the
    template delimits with double quotes) and must survive, or "he'd" is synthesized as "hed";
    typographic apostrophes are folded to ASCII so the tokenizer sees one consistent form.
    """
    v = str(value or "").replace("\u2019", "'").replace("\u2018", "'")
    v = re.sub(r"[" + _AUK_DQ + r"]", "", v)
    return re.sub(r"\s+", " ", v).strip()


def _auk_format_text(text: str) -> str:
    """Normalize the target text: drop quotes, collapse whitespace, ensure terminal punctuation."""
    t = _auk_strip_slot(text)
    if not t:
        return ""
    if t[-1] not in ".!?。！？…":
        t += "。" if _auk_text_language(t) == "zh" else "."
    return t


def _auk_build_instruct(style: str, text: str) -> str:
    """Render the official Instruct-TTS template. Template language follows the *spoken text*.

    pe.config.yaml sets `output_language: text_language` for instruct_tts, i.e. the template is
    chosen by the language of the words being spoken, not by the language of the description.
    A Chinese description driving English text is explicitly supported (see the AuK COOKBOOK
    Instruct TTS example and the official funeral demo).
    """
    t = _auk_format_text(text)
    s = _auk_strip_slot(style)
    if not s:
        s = "a clear, natural, expressive voice with balanced tone and fluent pacing"
    return AUK_INSTRUCT_TTS_TEMPLATE[_auk_text_language(t)].format(style=s, text=t)


# Automatic Instruction Normalizer & Spoken Text Extractor
def normalize_auk_instruction(instruction: str, has_audio: bool = False, task_type: str = None):
    """Rewrite any supported phrasing into the official AuK template.

    Returns (canonical_instruction, spoken_text, style_desc). `style_desc` is "" for tasks that
    carry no voice description, and is used by estimate_auto_duration() to read timing cues.

    Every Instruct-TTS branch emits AUK_INSTRUCT_TTS_TEMPLATE verbatim. In particular the
    demo phrasing `Say the following in the voice described here: "...", and say: "..."` is an
    *input to the Prompt Enhancer*, not a model prompt -- infer_gradio.py's demo picker sets
    use_pe=True alongside it. Passed to the model raw, its leading quoted span reads as content
    and gets spoken, which is exactly the Chinese-leakage failure it produces.
    """
    inst = str(instruction or "").strip().replace("—", "-").replace("–", "-")
    q = _AUK_Q
    dq = _AUK_DQ

    # --- Editing tasks: the instruction is already an official model template -----------------
    # Hand it to the model byte-for-byte. The previous keyword heuristic decided this by looking
    # for words like "emotion" or "replace" in the text, which silently misfired: "Change the
    # emotion to sad." contains none of them, so it fell through to the bare-text branch and,
    # with reference audio attached, was rewritten into
    #     Say the following with the same voice: "Change the emotion to sad."
    # i.e. the emotion edit became a voice clone that SPOKE the instruction. Dispatching on the
    # selected task instead of on keywords removes that whole class of failure.
    strategy = TASK_TEMPLATES.get(task_type, {}).get("duration") if task_type else None
    if strategy and strategy != "tts_by_text":
        quoted = re.findall(rf"[{q}](.*?)[{q}]", inst, re.DOTALL)
        return inst, (quoted[-1].strip() if quoted else ""), ""

    # --- Instruct TTS: style first, then text -------------------------------------------------
    two_slot_patterns = (
        # Prompt-Enhancer / demo phrasing
        r"Say the following in the voice described here:\s*[{q}](?P<style>.*?)[{q}]\s*[,，]?\s*and say:\s*[{q}](?P<text>.*)[{q}]?\s*\.?\s*$",
        # Official EN template (pe.config.yaml tasks.instruct_tts.templates.en)
        r"Based on the following description:\s*[{q}](?P<style>.*?)[{q}]\s*[,，]\s*generate speech content\s*[{q}](?P<text>.*?)[{q}]?\s*\.?\s*$",
        # COOKBOOK EN variant
        r"Generate speech based on the following description:\s*[{q}](?P<style>.*?)[{q}]\s*\.?\s*The content to speak is:\s*[{q}](?P<text>.*?)[{q}]?\s*\.?\s*$",
        # Official ZH template (pe.config.yaml tasks.instruct_tts.templates.zh)
        r"请基于下面的描述[:：]\s*[{q}](?P<style>.*?)[{q}]\s*[,，]?\s*生成语音内容\s*[{q}](?P<text>.*?)[{q}]?\s*[.。]?\s*$",
    )
    for pattern in two_slot_patterns:
        m = re.search(pattern.format(q=q), inst, re.DOTALL | re.IGNORECASE)
        if m:
            style, text = m.group("style").strip(), m.group("text").strip()
            return _auk_build_instruct(style, text), _auk_format_text(text), _auk_strip_slot(style)

    # --- Zero-shot TTS (voice cloning) --------------------------------------------------------
    m = re.search(
        rf"Say the following (?:with the same voice|in the reference speaker's voice):\s*[{q}](?P<text>.*?)[{q}]?\s*\.?\s*$",
        inst,
        re.DOTALL | re.IGNORECASE,
    )
    if m:
        text = _auk_format_text(m.group("text"))
        return AUK_ZERO_SHOT_TTS_TEMPLATE.format(text=text), text, ""

    # --- FINETUNING.md phrasing: Say "{text}" in {style} --------------------------------------
    # Accepted as input, but never forwarded as-is: a single `Say "<text>" in <style>` sentence
    # is one continuous utterance to the DiT, so the trailing style gets voiced once the text
    # runs out. Re-slot it into the official template instead.
    m = re.search(
        rf"^Say\s*[{dq}](?P<text>.*)[{dq}]\s*(?:in|with)\s+(?P<style>.+?)\s*$",
        inst,
        re.DOTALL | re.IGNORECASE,
    )
    if m:
        style, text = m.group("style").strip(), m.group("text").strip()
        style = re.sub(r"^(?:a|an|the)\s+", "", style, flags=re.IGNORECASE)
        style = re.sub(r"^voice\s+(?=as if|like|that)", "", style, flags=re.IGNORECASE)
        return _auk_build_instruct(style, text), _auk_format_text(text), _auk_strip_slot(style)

    # --- Bare text -----------------------------------------------------------------------------
    editing_keywords = [
        "say the following", "voice described", "replace", "change", "singing",
        "lyrics", "volume", "pitch", "speed", "emotion", "timbre", "accent",
        "whisper", "noise", "clean", "vocal", "separate", "based on the following",
        "generate speech based on", "the content to speak is", "semitone", "decibel",
        "remove the", "add a", "extract the", "keep the",
    ]
    if not any(k in inst.lower() for k in editing_keywords):
        if has_audio:
            text = _auk_format_text(inst)
            return AUK_ZERO_SHOT_TTS_TEMPLATE.format(text=text), text, ""
        return _auk_build_instruct("", inst), _auk_format_text(inst), ""

    # --- Editing tasks: pass through untouched -------------------------------------------------
    quoted = re.findall(rf"[{q}](.*?)[{q}]", inst, re.DOTALL)
    return inst, (quoted[-1].strip() if quoted else ""), ""


# Per-task duration rules (pe.config.yaml -> tasks.<task>.duration)
#
# Verified against the official demo durations in infer_gradio.py DEMO_EXAMPLE_GROUPS by
# measuring the bundled input wavs: the emotion multipliers reproduce all four emotion demos to
# within 1.5%, and speed_scaled reproduces all five speed demos to within 6.8%. The residual is
# the silero VAD trim the Prompt Enhancer applies to its *input* before measuring; this notebook
# feeds the model untrimmed audio, so matching the untrimmed source length is the consistent
# choice here (and avoids a silero_vad dependency).
AUK_EMOTION_MULTIPLIERS = {
    "sad": 1.22, "悲伤": 1.22, "难过": 1.22,
    "fearful": 1.16, "afraid": 1.16, "scared": 1.16, "fear": 1.16, "害怕": 1.16, "恐惧": 1.16,
    "happy": 1.06, "开心": 1.06, "高兴": 1.06,
    "angry": 1.06, "生气": 1.06, "愤怒": 1.06,
    "surprised": 1.06, "惊讶": 1.06,
    "disgusted": 1.06, "厌恶": 1.06,
    "calm": 1.06, "平静": 1.06,
    "excited": 1.06, "兴奋": 1.06,
}
AUK_DEFAULT_EMOTION_MULTIPLIER = 1.06

# (keywords, seconds added when inserting, seconds removed when deleting)
AUK_NONVERBAL_ADJUSTMENTS = (
    (("呼吸", "换气", "喘", "breath", "breathing", "pant", "inhale", "exhale"), 0.35, -0.6),
    (("咂嘴", "咂舌", "吸鼻", "倒吸", "惊讶", "tsk", "smack", "sniff", "gasp"), 0.5, -1.0),
    (("笑", "叹气", "咳", "清嗓", "语气", "laugh", "laughter", "chuckle", "sigh", "cough", "throat", "clearing"), 0.75, -1.05),
)
AUK_DEFAULT_NONVERBAL = (0.55, -0.9)


def _auk_audio_seconds(path: str) -> float:
    """Duration of an audio file in seconds, or 0.0 only if it genuinely cannot be read.

    torchaudio.info() raises on some containers depending on which backend is installed. That
    used to be swallowed by a bare `except`, leaving ref_seconds at 0 so every source-relative
    task fell back to the 3.0 s placeholder no matter how long the input really was -- the
    "input is 6 s, output is 3 s" failure. Fall back to a full decode, and say so when both fail.
    """
    if not path or not os.path.isfile(path):
        return 0.0
    try:
        info = torchaudio.info(path)
        if info.num_frames > 0 and info.sample_rate > 0:
            return float(info.num_frames) / float(info.sample_rate)
    except Exception as exc:
        print(f"[AuK Engine] torchaudio.info failed on {path} ({exc!r}); decoding to measure it.", flush=True)
    try:
        wav, sr = torchaudio.load(path)
        if wav.shape[-1] > 0 and sr > 0:
            return float(wav.shape[-1]) / float(sr)
    except Exception as exc:
        print(f"[AuK Engine] WARNING: cannot read the duration of {path} ({exc!r}).", flush=True)
    return 0.0


def _auk_spoken_duration(text: str, fallback_language: str = "en") -> float:
    """Port of pe._spoken_duration: rough spoken length of a fragment, for content scaling."""
    value = str(text or "")
    num_zh = len(_AUK_CJK_RE.findall(value))
    num_en = len(_AUK_EN_WORD_RE.findall(value))
    duration = num_zh * 0.21 + num_en * 0.30
    if duration > 0:
        return duration
    language = _auk_text_language(value, fallback_language)
    units = len(value.split()) if language == "en" else len(re.findall(r"\S", value))
    return units * (0.30 if language == "en" else 0.21)


def _auk_parse_edit_slots(instruction: str) -> tuple[str, str]:
    """Extract (added_text, deleted_text) from a content / lyric edit instruction."""
    q = _AUK_Q
    patterns = (
        # Replace "orig" with "new"  ->  groups are (delete, add)
        (rf"(?:replace|change|swap)\s*[{q}](.*?)[{q}]\s*(?:with|to|for)\s*[{q}](.*?)[{q}]", "da"),
        (rf"[把将]\s*[{q}](.*?)[{q}].*?(?:改成|换成|替换为|改为)\s*[{q}](.*?)[{q}]", "da"),
        # Insert / add "new" before|after ...
        (rf"(?:insert|add)\s*[{q}](.*?)[{q}]\s*(?:before|after|at)", "a"),
        (rf"在.*?(?:前面|后面|之前|之后)\s*(?:插入|加上|添加)\s*[{q}](.*?)[{q}]", "a"),
        # Delete / remove "orig"
        (rf"(?:delete|remove|drop)\s*(?:the\s+)?(?:word|words|phrase|line)?\s*[{q}](.*?)[{q}]", "d"),
        (rf"(?:删除|删掉|去掉|移除)\s*[{q}](.*?)[{q}]", "d"),
    )
    for pattern, kind in patterns:
        m = re.search(pattern, instruction, re.IGNORECASE | re.DOTALL)
        if m:
            if kind == "da":
                return m.group(2).strip(), m.group(1).strip()
            return (m.group(1).strip(), "") if kind == "a" else ("", m.group(1).strip())
    return "", ""


def _auk_content_scaled(instruction: str, base_duration: float) -> float:
    """Port of pe._content_scaled_duration for the no-ASR case.

    With no transcript of the source audio we cannot know the original total, so a replace is
    scaled by the new/old fragment ratio (exactly what the official function does without ASR)
    and a pure insert/delete shifts the length by the fragment's own spoken duration.
    """
    added, deleted = _auk_parse_edit_slots(instruction)
    if not added and not deleted:
        return base_duration
    language = _auk_text_language(added or deleted)
    add_sec = _auk_spoken_duration(added, language) if added else 0.0
    del_sec = _auk_spoken_duration(deleted, language) if deleted else 0.0
    if added and deleted:
        return base_duration * add_sec / del_sec if del_sec > 0 else base_duration
    return max(0.4, base_duration + add_sec - del_sec)


def _auk_nonverbal_delta(instruction: str) -> float:
    """Port of pe._nonverbal_delta: seconds to add or remove for a nonverbal sound edit."""
    s = str(instruction or "").casefold()
    is_delete = bool(re.search(r"\b(?:remove|delete|drop|strip|take out)\b", s) or re.search(r"去掉|删除|删掉|移除", s))
    for keywords, add_sec, del_sec in AUK_NONVERBAL_ADJUSTMENTS:
        if any(k.casefold() in s for k in keywords):
            return del_sec if is_delete else add_sec
    add_sec, del_sec = AUK_DEFAULT_NONVERBAL
    return del_sec if is_delete else add_sec


def _auk_emotion_multiplier(instruction: str) -> float:
    """Port of pe emotion_multipliers: sad stretches most, everything else is a mild 1.06."""
    s = str(instruction or "").casefold()
    for name, multiplier in AUK_EMOTION_MULTIPLIERS.items():
        if name.isascii():
            if re.search(rf"\b{re.escape(name)}\b", s):
                return multiplier
        elif name in s:
            return multiplier
    return AUK_DEFAULT_EMOTION_MULTIPLIER


# Automatic Duration Estimator (port of the AuK Prompt Enhancer duration strategy)
def estimate_auto_duration(
    instruction: str,
    audio_path: str = None,
    task_type: str = None,
    style_desc: str = None,
) -> float:
    """Target length in seconds for gen_seconds, for every task in TASK_TEMPLATES.

    Dispatches on the task's `duration` strategy, mirroring pe._compute_duration:
      tts_by_text     F5 text baseline x timing-cue ratio (no reference audio needed)
      equal_length    match the source clip
      speed_scaled    source / speed multiplier
      emotion_scaled  source x emotion coefficient
      content_scaled  source adjusted by the inserted/deleted words
      nonverbal_delta source +/- a fixed offset for the nonverbal sound

    It deliberately never pads: for TTS, unused frames are what the model fills by speaking the
    style description; for editing, over-long targets stretch or repeat the source.
    """
    ref_seconds = _auk_audio_seconds(audio_path)
    strategy = TASK_TEMPLATES.get(task_type, {}).get("duration", "")
    if not strategy:
        # No task selected (or a custom instruction): infer from whether audio is attached.
        strategy = "equal_length" if ref_seconds > 0 else "tts_by_text"

    if strategy == "tts_by_text":
        canonical, text_to_speak, parsed_style = normalize_auk_instruction(
            instruction, has_audio=ref_seconds > 0, task_type=task_type
        )
        if style_desc is None:
            style_desc = parsed_style
        if text_to_speak.strip():
            baseline = _auk_f5_baseline(text_to_speak, _auk_text_language(text_to_speak))
            return _auk_quantize_seconds(min(max(baseline * _auk_pace_ratio(style_desc), 0.4), 60.0))
        # Nothing to speak -> fall through to the reference length if we have one.
        return _auk_quantize_seconds(max(1.0, ref_seconds)) if ref_seconds > 0 else 3.0

    if ref_seconds <= 0:
        # Every remaining strategy is relative to the source clip, so without a readable input
        # there is nothing to scale. Say so rather than quietly emitting a 3 s stub.
        print(
            f"[AuK Engine] WARNING: '{task_type}' uses the '{strategy}' duration rule, which needs the "
            f"input clip's length, but none could be read from {audio_path!r}. Falling back to 3.00s -- "
            f"set Target Duration manually to match your input.",
            flush=True,
        )
        return 3.0

    if strategy == "speed_scaled":
        match = re.search(r"(\d+(?:\.\d+)?)\s*[xX倍]", instruction)
        factor = float(match.group(1)) if match else 0.0
        return _auk_quantize_seconds(max(0.4, ref_seconds / factor if factor > 0 else ref_seconds))

    if strategy == "emotion_scaled":
        return _auk_quantize_seconds(max(0.4, ref_seconds * _auk_emotion_multiplier(instruction)))

    if strategy == "content_scaled":
        return _auk_quantize_seconds(max(0.4, _auk_content_scaled(instruction, ref_seconds)))

    if strategy == "nonverbal_delta":
        return _auk_quantize_seconds(max(0.4, ref_seconds + _auk_nonverbal_delta(instruction)))

    # equal_length
    return _auk_quantize_seconds(max(0.4, ref_seconds))


# Task Templates Catalog
#
# Every `template` below is the OFFICIAL model-facing string from pe.config.yaml
# (tasks.<task>.templates) -- NOT the phrasing shown in the AuK demo gallery. The demo strings
# in infer_gradio.py are Prompt-Enhancer inputs (the picker sets use_pe=True beside them); the
# PE rewrites them into these templates before the DiT ever sees them. Feeding a demo string
# straight to the model is why several editing tasks silently no-opped.
#
# `lang` records which language variant the model actually receives. Several tasks ship a zh
# template only; pe.py resolves a missing variant with
#     templates.get(language) or templates.get("en") or templates.get("zh")
# so an English request for those tasks still reaches the model in Chinese. Those templates are
# left in Chinese here for exactly that reason -- translating them breaks the task.
TASK_TEMPLATES = {
    "Instruct TTS - Funeral Speech (Devastating News)": {
        "desc": "Solemn emotional speech: choked funeral voice delivering devastating news with heavy pauses and grief.",
        "template": 'Based on the following description: "A speaker delivering devastating news at a funeral: a soft, choked voice forcing back tears, speech breaking between pauses, each word heavier than the last, the final line torn open by grief, trembling with a sob that can no longer be held down. Slow, low intonation with unsteady breath; clarity slightly eroded by the choking. The overall mood is solemn and mournful.", generate speech content "He always said he\'d come back. He always kept his word. Until now.".',
        "duration": "tts_by_text",
        "lang": "en",
        "needs_audio": False,
        "sample_audio": None,
    },
    "Instruct TTS - Shakespearean Villain (Dramatic Baritone)": {
        "desc": "Theatrical villain revealing the truth: dramatic baritone with rolling consonants and sinister poise.",
        "template": 'Based on the following description: "A classic Shakespearean theatrical villain savouring the moment before revealing the truth: a rich, dramatic baritone with deep chest resonance, measured and deliberate pacing with pointed pauses, exaggerated rolling consonants and stage-clear diction. The tone begins mockingly sweet and tender, then turns knife-sharp with dangerous delight.", generate speech content "You see, my dear, the trap was never meant for you. It was always meant for him.".',
        "duration": "tts_by_text",
        "lang": "en",
        "needs_audio": False,
        "sample_audio": None,
    },
    "Instruct TTS - Expressive Narration (Custom Voice)": {
        "desc": "Custom instruction TTS -- edit the description and the spoken text to taste.",
        "template": 'Based on the following description: "A confident, warm radio host with a resonant baritone, clear articulation and an even, professional delivery.", generate speech content "Welcome to the world of AuK, an open-source foundational speech model.".',
        "duration": "tts_by_text",
        "lang": "en",
        "needs_audio": False,
        "sample_audio": None,
    },
    "Zero-shot TTS (Voice Cloning)": {
        "desc": "Speak new text in the voice of the reference clip.",
        "template": 'Say the following with the same voice: "Ladies and gentlemen, it is an honor to speak before this audience."',
        "duration": "tts_by_text",
        "lang": "en",
        "needs_audio": True,
        "sample_audio": "AuK/assets/demo-input-audio/zero-shot-tts/ref.wav",
    },
    "Speech Content Editing (Replace)": {
        "desc": "Replace words in the audio. Official template uses single quotes and a trailing period.",
        "template": "Replace 'but accepting what we cannot have' with 'and living well with dreams unmet'.",
        "duration": "content_scaled",
        "lang": "en",
        "needs_audio": True,
        "sample_audio": "AuK/assets/demo-input-audio/content-edit/content.wav",
    },
    "Speech Content Editing (Insert)": {
        "desc": "Insert new words before or after an anchor phrase. Inserting lengthens the clip.",
        "template": "Add 'and quietly' before 'accepting what we cannot have'.",
        "duration": "content_scaled",
        "lang": "en",
        "needs_audio": True,
        "sample_audio": "AuK/assets/demo-input-audio/content-edit/content.wav",
    },
    "Speech Content Editing (Delete)": {
        "desc": "Remove a phrase from the audio. Deleting shortens the clip.",
        "template": "Remove 'but accepting what we cannot have'.",
        "duration": "content_scaled",
        "lang": "en",
        "needs_audio": True,
        "sample_audio": "AuK/assets/demo-input-audio/content-edit/content.wav",
    },
    "Lyric Editing (Singing Voice)": {
        "desc": "Change words in an a cappella vocal recording.",
        "template": 'Change "rear view" to "like you" in the vocal recording.',
        "duration": "content_scaled",
        "lang": "en",
        "needs_audio": True,
        "sample_audio": "AuK/assets/demo-input-audio/vocal-edit/vocaledit-en-1-input.wav",
    },
    "Pitch Editing": {
        "desc": "Raise or lower pitch by semitones.",
        "template": "Raise the pitch by 2 semitones.",
        "duration": "equal_length",
        "lang": "en",
        "needs_audio": True,
        "sample_audio": "AuK/assets/demo-input-audio/pitch/pitch-1-input.wav",
    },
    "Speed Editing": {
        "desc": "Adjust speaking rate (0.5x / 0.75x / 1.25x / 1.5x / 2.0x).",
        "template": "Adjust the speech speed to 1.25x.",
        "duration": "speed_scaled",
        "lang": "en",
        "needs_audio": True,
        "sample_audio": "AuK/assets/demo-input-audio/speed/speed-edit-1-input.wav",
    },
    "Volume Editing": {
        "desc": "Increase or decrease loudness in decibels.",
        "template": "Increase the volume by 5 dB.",
        "duration": "equal_length",
        "lang": "en",
        "needs_audio": True,
        "sample_audio": "AuK/assets/demo-input-audio/energy/energy-edit-1-input.wav",
    },
    "Emotion Editing": {
        "desc": "Change the emotion (happy / sad / angry / surprised / calm / excited / fearful).",
        "template": "Change the emotion to sad.",
        "duration": "emotion_scaled",
        "lang": "en",
        "needs_audio": True,
        "sample_audio": "AuK/assets/demo-input-audio/emotion-edit/en-1-input.wav",
    },
    "Timbre Editing": {
        "desc": "Keep the words, change the voice to match a description.",
        "template": 'Keep the spoken content unchanged and change the timbre to: "a deep, calm male voice with clear articulation".',
        "duration": "equal_length",
        "lang": "en",
        "needs_audio": True,
        "sample_audio": "AuK/assets/demo-input-audio/vc/vc-1-input.wav",
    },
    "De-accenting (Dialect Removal)": {
        "desc": "Remove a regional accent, keeping the speaker's timbre. Chinese-only template.",
        "template": "请去掉这段语音里的方言口音，保持说话人音色一致。",
        "duration": "equal_length",
        "lang": "zh",
        "needs_audio": True,
        "sample_audio": "AuK/assets/demo-input-audio/accent/accent-anhui-input.wav",
    },
    "Nonverbal - Add Sound (EN)": {
        # English phrasing, verbatim from the AuK demo gallery. Note pe.config.yaml ships NO
        # English template for nonverbal_edit add_* (only for delete), so the Prompt Enhancer
        # would translate this into the Chinese form before the model saw it. English is the
        # default here because that is what the gallery shows users typing; if the sound does
        # not appear, try the "(ZH template)" preset, which is the exact string the PE emits.
        # The anchor must be words that really occur in the clip: both "We tested" and
        # "only one of them" occur in en-c-input.wav.
        "desc": "Insert a breath / sneeze / pause / laugh at an anchor phrase. English demo phrasing.",
        "template": 'Add a breath before "We tested"',
        "duration": "nonverbal_delta",
        "lang": "en",
        "needs_audio": True,
        "sample_audio": "AuK/assets/demo-input-audio/nv/en-c-input.wav",
    },
    "Nonverbal - Add Sound (ZH template)": {
        "desc": "Same edit through the official Chinese template - the exact string the Prompt Enhancer emits. Fallback if the English form does nothing.",
        "template": "在“We tested”前增加呼吸声。",
        "duration": "nonverbal_delta",
        "lang": "zh",
        "needs_audio": True,
        "sample_audio": "AuK/assets/demo-input-audio/nv/en-c-input.wav",
    },
    "Nonverbal - Remove Sound (EN)": {
        "desc": "Strip a nonverbal sound (humming, hiss, sobbing, laughter). This IS the official English template.",
        "template": "Remove all the humming from the audio.",
        "duration": "nonverbal_delta",
        "lang": "en",
        "needs_audio": True,
        "sample_audio": "AuK/assets/demo-input-audio/nv/en-d-input.wav",
    },
    "Normal Voice -> Whisper": {
        # wh-w2n-zh-input.wav is NORMAL voiced speech, not a whisper: measured 73% voiced frames
        # with a 177 Hz F0, more strongly voiced than the known-normal control clips (a whisper
        # has no periodic excitation at all). That matches the AuK gallery, which pairs this file
        # with "Turn this into a whisper" -- the file is the INPUT to whisper conversion.
        "desc": "Convert normal speech to a whisper. Chinese-only template.",
        "template": "用小声耳语的方式把这段话说出来。",
        "duration": "equal_length",
        "lang": "zh",
        "needs_audio": True,
        "sample_audio": "AuK/assets/demo-input-audio/whisper/wh-w2n-zh-input.wav",
    },
    "Whisper -> Normal Voice": {
        # No whispered clip ships with AuK, so this preset has no bundled sample: upload your own
        # whispered recording, or make one with the preset above and feed the result back in.
        "desc": "Restore normal phonation from a whisper. Chinese-only template. UPLOAD your own whispered clip - AuK ships no whispered sample.",
        "template": "把这段耳语转换成正常说话的声音。",
        "duration": "equal_length",
        "lang": "zh",
        "needs_audio": True,
        "sample_audio": None,
    },
    "Speech Enhancement & Denoising": {
        "desc": "Denoise and dereverberate, keeping every speaker. Chinese-only template.",
        "template": "请清理这段输入语音，不做说话人删除，并去除房间混响，输出干净的人声结果。",
        "duration": "equal_length",
        "lang": "zh",
        "needs_audio": True,
        "sample_audio": "AuK/assets/demo-input-audio/se/se-zh-1-input.wav",
    },
    "Speaker Separation": {
        "desc": "Keep only one speaker and drop the rest. Chinese-only template; quote words that speaker says.",
        "template": "请只保留说'get what'的人，去掉其他说话人，输出等长纯净人声。",
        "duration": "equal_length",
        "lang": "zh",
        "needs_audio": True,
        "sample_audio": "AuK/assets/demo-input-audio/ss/en-1-input.wav",
    },
    "Vocal Extraction (Remove Accompaniment)": {
        "desc": "Keep the singing voice, drop the instrumental.",
        "template": "Keep only the singing voice, drop everything else.",
        "duration": "equal_length",
        "lang": "en",
        "needs_audio": True,
        "sample_audio": "AuK/assets/demo-input-audio/vocal-extraction/vocal-1-input.wav",
    },
}

# Generator core function using official tested pipeline
@torch.inference_mode()
def run_auk_inference(audio_path, instruction, gen_seconds=0.0, seed=-1, task_type=None, nfe=None, cfg=None,
                      normalize_level=True, lowpass_hf=False):
    start_time = time.time()
    if not instruction or not instruction.strip():
        raise gr.Error("Please enter an instruction.")

    instruction = instruction.strip()
    has_audio = audio_path and os.path.isfile(audio_path)

    # Check if this task requires audio and auto-fallback to bundled sample if missing
    task_info = TASK_TEMPLATES.get(task_type, {})
    if not has_audio and task_info.get("needs_audio", False):
        sample_file = task_info.get("sample_audio")
        if sample_file and os.path.isfile(sample_file):
            print(f"[AuK Engine] No audio uploaded for '{task_type}'. Auto-loading bundled sample audio: {sample_file}", flush=True)
            audio_path = sample_file
            has_audio = True
        else:
            raise gr.Error(f"Task '{task_type}' requires reference audio. Please upload an audio file or select an Instruct TTS preset.")

    # The UI dices the seed into the Seed box before calling this, so use it verbatim; a
    # negative value only reaches here on a direct call, and still means "pick one for me".
    effective_seed = random.randint(1, 2147483647) if (seed is None or int(seed) < 0) else int(seed)

    # Rewrite into the official AuK template and split out the two slots
    canonical_instruction, spoken_text, style_desc = normalize_auk_instruction(
        instruction, has_audio=has_audio, task_type=task_type
    )
    print(f"[AuK Engine] Normalized Prompt: {canonical_instruction}", flush=True)
    if spoken_text:
        print(f"[AuK Engine] Spoken text (the ONLY words that should be voiced): '{spoken_text}'", flush=True)
    if style_desc:
        print(f"[AuK Engine] Style conditioning (must never be voiced): '{style_desc[:90]}...'", flush=True)

    content = [{"type": "text", "text": canonical_instruction}]
    if has_audio:
        content.append({"type": "audio", "audio": audio_path})

    messages = [{"role": "user", "content": content}]

    # Compute effective target duration (Auto or Manual)
    auto_seconds = estimate_auto_duration(canonical_instruction, audio_path, task_type, style_desc=style_desc)
    if gen_seconds is None or float(gen_seconds) <= 0.0:
        effective_seconds = auto_seconds
        is_auto = True
    else:
        effective_seconds = _auk_quantize_seconds(float(gen_seconds))
        is_auto = False
        # AuK has no EOS: any duration beyond the natural length of the target text is filled by
        # decoding the style description aloud. Warn before that happens rather than after.
        if spoken_text and effective_seconds > auto_seconds * 1.25:
            print(
                f"[AuK Engine] WARNING: manual duration {effective_seconds:.2f}s exceeds the estimated "
                f"natural length {auto_seconds:.2f}s by {100 * (effective_seconds / auto_seconds - 1):.0f}%. "
                f"AuK will fill the surplus frames by speaking the style description. "
                f"Set Duration to 0 for auto, or lower it to ~{auto_seconds:.2f}s.",
                flush=True,
            )

    # AuK-Flash re-pins these inside AukInfer.generate(), so UI values only matter for base.
    eff_nfe = AUK_NFE if (nfe is None or engine.is_flash) else int(nfe)
    eff_cfg = AUK_CFG if (cfg is None or engine.is_flash) else float(cfg)

    print(f"\n[AuK Engine] Starting task: '{task_type}' | Target Duration: {effective_seconds:.2f}s ({'Auto' if is_auto else 'Manual'}) | Steps: {eff_nfe} | CFG: {eff_cfg} | Seed: {effective_seed}", flush=True)

    # Call official generate pipeline with effective_seed
    audio_out, sr = engine.generate(
        messages,
        audio=audio_path if has_audio else None,
        gen_seconds=effective_seconds,
        nfe=eff_nfe,
        cfg_strength=eff_cfg,
        seed=effective_seed,
    )

    waveform = audio_out.detach().to(torch.float32).cpu()
    if waveform.ndim == 2 and 1 in waveform.shape:
        waveform = waveform.reshape(-1)
    elif waveform.ndim == 2:
        waveform = waveform.mean(dim=0)

    raw_amp = float(torch.max(torch.abs(waveform)))
    mean_amp = float(torch.mean(torch.abs(waveform)))
    raw_dbfs = 20 * math.log10(raw_amp) if raw_amp > 0 else -120.0
    print(f"[AuK Engine] Audio stats: Peak = {raw_amp:.4f} ({raw_dbfs:+.1f} dBFS) | Mean = {mean_amp:.4f}", flush=True)

    # Tame near-Nyquist hash from the BigVGAN decode. At 24 kHz the band above 10.5 kHz carries
    # well under 1% of a speech clip's energy, but the vocoder intermittently puts a burst of
    # sample-alternating oscillation up there, which reads as crackle / "clipping" on playback.
    if lowpass_hf:
        waveform = torchaudio.functional.lowpass_biquad(waveform, sr, cutoff_freq=10500.0, Q=0.707)

    # Level. AuK's raw output commonly peaks around -11 dBFS; left alone it invites turning the
    # playback gain up, which amplifies exactly the artifacts above. Normalizing to -1 dBFS keeps
    # a little headroom and never hard-clips. Without it, only attenuate a hot signal.
    max_amp = float(torch.max(torch.abs(waveform)))
    if normalize_level and max_amp > 0:
        waveform = waveform * (10 ** (-1.0 / 20.0) / max_amp)
    elif max_amp > 0.95:
        waveform = waveform * (0.95 / max_amp)

    # Export clean uncompressed 16-bit Linear PCM WAV for 100% universal browser and player playback
    waveform_clamped = torch.clamp(waveform, -1.0, 1.0)
    out_filename = "generated_output.wav"
    torchaudio.save(out_filename, waveform_clamped.unsqueeze(0), sr, encoding="PCM_S", bits_per_sample=16)

    elapsed = time.time() - start_time
    output_duration = len(waveform) / sr
    print(f"[AuK Engine] Generation complete in {elapsed:.2f}s | Output Duration: {output_duration:.2f}s (RTF: {elapsed / max(0.1, output_duration):.2f})!", flush=True)

    return out_filename, output_duration, elapsed, is_auto, effective_seconds, raw_amp, effective_seed, eff_nfe


# Gradio Custom Styling
AIQUEST_CSS = """
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;600;700&display=swap');
* { font-family: 'Inter', sans-serif !important; }
.gradio-container { max-width: 1000px !important; margin: auto !important; }
.brand-header { text-align: center; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 28px; border-radius: 15px; margin-bottom: 20px; box-shadow: 0 10px 25px rgba(102,126,234,0.3); }
.brand-title { color: white; font-size: 2em; font-weight: 700; margin: 0 0 6px 0; }
.brand-subtitle { color: rgba(255,255,255,0.88); font-size: 1em; margin-bottom: 16px; }
.social-buttons { display: flex; justify-content: center; gap: 12px; flex-wrap: wrap; }
.social-btn { padding: 10px 24px; border-radius: 8px; font-weight: 700; font-size: 15px; text-decoration: none; display: inline-block; color: white; transition: all 0.3s; box-shadow: 0 4px 12px rgba(0,0,0,0.2); }
.social-btn:hover { transform: translateY(-2px); box-shadow: 0 6px 16px rgba(0,0,0,0.3); }
.youtube-btn { background: linear-gradient(135deg, #FF0000 0%, #CC0000 100%); }
.x-btn { background: linear-gradient(135deg, #000000 0%, #333333 100%); }
button.primary { background: linear-gradient(135deg, #667eea 0%, #764ba2 100%) !important; color: white !important; font-weight: 600 !important; border-radius: 12px !important; }
#stop-btn { background: linear-gradient(135deg, #ef4444 0%, #b91c1c 100%) !important; color: white !important; font-weight: 600 !important; border-radius: 12px !important; }
#clear-btn { background: linear-gradient(135deg, #6b7280 0%, #374151 100%) !important; color: white !important; font-weight: 600 !important; border-radius: 12px !important; }
.footer { text-align: center; padding: 20px; margin-top: 30px; border-top: 2px solid #e5e7eb; color: #6b7280; }
"""

# Build Gradio Blocks App
with gr.Blocks(title="AuK - AIQUEST Academy", css=AIQUEST_CSS) as demo:
    gr.HTML("""
    <div class="brand-header">
        <div class="brand-title">🎤 AuK v1.0 - Unified Audio Generation and Editing</div>
        <div class="brand-subtitle">Kaggle T4 x2 GPU Edition - Powered by <strong>AIQUEST Academy</strong></div>
        <div class="social-buttons">
            <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1" target="_blank" class="social-btn youtube-btn">▶ Subscribe on YouTube</a>
            <a href="https://x.com/aiquestacademy" target="_blank" class="social-btn x-btn">𝕏 Follow on X</a>
        </div>
    </div>
    """)

    # Single source of truth for the preset the UI opens on -- keep the dropdown value and the
    # instruction textbox derived from the same key so renaming a preset cannot desync them.
    DEFAULT_TASK = next(iter(TASK_TEMPLATES))

    with gr.Row():
        with gr.Column(scale=1):
            task_selector = gr.Dropdown(
                choices=list(TASK_TEMPLATES.keys()),
                value=DEFAULT_TASK,
                label="🎯 Select Task Preset",
                info="Select official benchmark presets or custom editing/cloning tasks",
            )
            in_audio = gr.Audio(
                label="Input Reference Audio (Auto-loaded for presets, or upload your own)",
                type="filepath",
            )
            in_instruction = gr.Textbox(
                label="Natural Language Instruction",
                value=TASK_TEMPLATES[DEFAULT_TASK]["template"],
                lines=4,
            )
            with gr.Row():
                in_duration = gr.Slider(
                    minimum=0.0,
                    maximum=30.0,
                    step=0.5,
                    value=0.0,
                    label="Target Duration (Seconds) - 0 = Auto",
                    info=(
                        "Leave at 0. AuK has no end-of-speech token, so this value alone decides "
                        "the output length: set it longer than the text needs and the surplus gets "
                        "filled by speaking the style description aloud. Auto derives the length "
                        "from the text plus its timing cues."
                    ),
                )
            with gr.Row():
                in_seed = gr.Number(value=-1, label="Seed", precision=0, scale=3,
                                    info="The exact seed used for the last run. Re-run it by unchecking Random Seed.")
                btn_new_seed = gr.Button("🎲 Roll", scale=1, size="sm")
                in_randomize_seed = gr.Checkbox(value=True, label="🎲 Random Seed", scale=2,
                                               info="Dice a new seed into the box on every Generate.")
            with gr.Row():
                in_normalize = gr.Checkbox(
                    value=True, label="🔊 Normalize level",
                    info="Peak-normalize output to -1 dBFS. Raw AuK output often peaks near -11 dBFS.",
                )
                in_lowpass = gr.Checkbox(
                    value=False, label="✂️ Tame >10.5 kHz",
                    info="Low-pass away the near-Nyquist vocoder hash that reads as crackle. Costs a little air.",
                )
            with gr.Row():
                in_nfe = gr.Slider(
                    minimum=4,
                    maximum=48,
                    step=1,
                    value=AUK_NFE,
                    label="Sampling Steps (NFE)",
                    info="Base only. 32 is the official default; 16 halves the cost. Locked on AuK-Flash.",
                    interactive=not engine.is_flash,
                )
                in_cfg = gr.Slider(
                    minimum=0.0,
                    maximum=4.0,
                    step=0.1,
                    value=AUK_CFG,
                    label="CFG Strength",
                    info="Base only. 2.0 is the official default. Locked at 0 on AuK-Flash.",
                    interactive=not engine.is_flash,
                )

        with gr.Column(scale=1):
            gr.Markdown(f"""
            ### ⚡ Inference Profile: {variant_cfg['label']}
            - **Backbone**: 1.5B Flow-Matching DiT + Qwen2.5-Omni Thinker
            - **Precision**: Full FP32 DiT + BFloat16 Qwen (Prevents FP16 overflow & attention collapse)
            - **Sampling**: **{AUK_NFE} steps**, CFG={AUK_CFG}
            - **Hardware Acceleration**: Kaggle Dual T4 GPUs (`GPU T4 x2`)
            """)
            out_audio = gr.Audio(label="Generated Audio Output", type="filepath")
            out_status = gr.Textbox(label="Status / Diagnostics", value="Ready", lines=2)

    with gr.Row():
        gen_btn = gr.Button("🎬 Generate Audio", variant="primary", size="lg", elem_id="gen-btn")
        stop_btn = gr.Button("🛑 Stop", variant="secondary", size="lg", elem_id="stop-btn")
        clear_btn = gr.Button("🗑️ Clear", variant="secondary", size="lg", elem_id="clear-btn")

    # Update template and sample audio on task change
    def on_task_change(selected_task):
        task_info = TASK_TEMPLATES.get(selected_task, {})
        sample_path = task_info.get("sample_audio")
        sample_audio = sample_path if (sample_path and os.path.isfile(sample_path)) else None
        strategy = task_info.get("duration", "equal_length")
        return (
            task_info.get("template", ""),
            sample_audio,
            0.0,  # always reset to auto; the strategy below decides the actual length
            f"Selected task: {selected_task} - {task_info.get('desc', '')} (auto duration: {strategy})",
        )

    task_selector.change(
        fn=on_task_change,
        inputs=[task_selector],
        outputs=[in_instruction, in_audio, in_duration, out_status],
    )

    btn_new_seed.click(fn=lambda: random.randint(1, 2147483647), outputs=[in_seed])

    # Generation handler with full exception logging
    def generate_handler(audio, instruction, duration, seed, task_type, nfe, cfg, normalize_level, lowpass_hf):
        try:
            out_file, out_dur, elapsed, is_auto, eff_sec, max_amp, used_seed, used_nfe = run_auk_inference(
                audio, instruction, duration, seed, task_type, nfe, cfg, normalize_level, lowpass_hf
            )
            mode_str = f"Auto {eff_sec:.2f}s" if is_auto else f"{eff_sec:.2f}s (Manual override)"
            amp_note = "normalized" if normalize_level else "raw"
            seed_note = str(used_seed)
            status_msg = f"Done in {elapsed:.2f}s | Audio Duration: {out_dur:.2f}s [{mode_str}] | Seed: {seed_note} | Raw peak: {20 * math.log10(max_amp) if max_amp > 0 else -120:+.1f} dBFS ({amp_note}) | {variant_cfg['label']} - {used_nfe} steps"
            return out_file, status_msg
        except Exception as e:
            import traceback
            err_trace = traceback.format_exc()
            print(f"\n[AuK Engine ERROR] Generation failed:\n{err_trace}", flush=True)
            return None, f"Error: {str(e)}"

    # Roll the seed into the box BEFORE generating, so the number visibly changes on every
    # click and the box always shows exactly what produced the audio you are hearing.
    def roll_seed(current_seed, randomize):
        if randomize or current_seed is None or int(current_seed) < 0:
            return random.randint(1, 2147483647)
        return int(current_seed)

    seed_event = gen_btn.click(
        fn=roll_seed,
        inputs=[in_seed, in_randomize_seed],
        outputs=[in_seed],
    )
    gen_event = seed_event.then(
        fn=generate_handler,
        inputs=[in_audio, in_instruction, in_duration, in_seed, task_selector, in_nfe, in_cfg, in_normalize, in_lowpass],
        outputs=[out_audio, out_status],
    )
    stop_btn.click(fn=None, cancels=[seed_event, gen_event])
    clear_btn.click(
        fn=lambda: (None, "", 0.0, -1, True, None, "Cleared"),
        outputs=[in_audio, in_instruction, in_duration, in_seed, in_randomize_seed, out_audio, out_status],
    )


    gr.HTML("""
    <div class="footer">
        <p style='margin: 0; text-align: center;'>Crafted with ❤️ by <strong>AIQUEST Academy</strong>. Built for Kaggle Dual T4 GPU.</p>
    </div>
    """)

print("Launching Gradio interface (embedded preview disabled, live logs active)...", flush=True)
demo.queue(max_size=10).launch(
    share=True,
    inline=False,
    show_error=True,
    quiet=False,
)